## tl;dr

개방ID `1042688965`는 2024~2025년 서울 성북구의 대학원 레코드다. 2025년에는 10개 패널 76행이 남아 있으나 학생·입학·졸업·교원·직원·학과 수는 모두 0이며, 비제로 값은 구조적 학교 표식과 입학정원 6명·수업료 3,967,000원뿐이다.

KEDI 원본의 동일 지역·유형·제로 프로필 후보는 `성신여자대학교 AI세무·회계대학원`(2025 학교코드 `53068G42`)이다. 성신여대 공식 연혁과 학칙 개정 이력은 이 대학원이 2024년 9월 1일 폐지되었다고 명시한다. 따라서 이 ID는 **신원확정·폐교 후 잔존 ID**로 분류하고 활성기관 분석에서는 제외하는 것이 타당하다.

## Context & Methods

EDSS 제한 DB에서 해당 개방ID의 연도·패널·측정값·학과명을 직접 조회하고, 2024·2025 KEDI 학교 원본에서 서울 성북구 부설대학원 후보를 교차검증한다. 공식 폐지 시점은 성신여대 연혁과 학칙 개정 이력을 근거로 한다.

### Key Assumptions

- `0`, 결측, 해당 없음은 구분한다. 여기서 “활동값 0”은 실제 수치가 0으로 기록된 경우만 뜻한다.
- 학교수 같은 구조적 표식과 등록금·정원 잔존값은 재적·입학·졸업 등 운영 실적과 구분한다.
- KEDI의 2025년 `학교상태=기존`은 공식 2024-09-01 폐지 기록과 충돌하므로 최신성 오류로 해석한다.
- 원본 파일은 읽기 전용으로 사용하며 수정하지 않는다.

## Data

### 1. Load sources and helpers

In [1]:
from __future__ import annotations

import decimal
import re
import zipfile
import xml.etree.ElementTree as ET
from pathlib import Path

import duckdb
from IPython.display import Markdown, display

OPEN_ID = "1042688965"
DB = Path("/Users/joocheol/Documents/GitHub/edss/data/processed/edss/restricted/edss_all.duckdb")
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
KEDI_DIR = PROJECT_ROOT / "data/raw/kedi/higher_education_school"
OFFICIAL_HISTORY = "https://www.sungshin.ac.kr/main_kor/10918/subview.do"
OFFICIAL_RULES = "https://rule.sungshin.ac.kr/service/law/joHistoryContent.do?contentSeq=83688&endDate=20251023&seq=17&startDate=20090301"

DIMENSIONS = {
    "조사년도", "개방ID", "적용년도", "학기구분명", "수업연한명", "개설기간명",
    "학년명", "연령명", "성별명", "학위과정구분명", "학과명", "학과한글명", "학과상태명",
    "단과대학명", "교육부계열명", "학과계열구분명", "주야간계절구분명", "본분교명",
    "시도명", "지역명", "학교구분명", "학제유형명", "교원구분명", "산업체경력유무명",
}
NS = {"m": "http://schemas.openxmlformats.org/spreadsheetml/2006/main"}

def numeric_nonzero(value: object) -> bool:
    if value in (None, ""):
        return False
    try:
        return decimal.Decimal(str(value).replace(",", "")) != 0
    except decimal.InvalidOperation:
        return False

def column_index(reference: str) -> int:
    result = 0
    for letter in re.match(r"[A-Z]+", reference).group(0):
        result = result * 26 + ord(letter) - 64
    return result - 1

def xlsx_records(path: Path) -> list[dict[str, str]]:
    with zipfile.ZipFile(path) as archive:
        shared_root = ET.fromstring(archive.read("xl/sharedStrings.xml"))
        shared = [
            "".join(node.text or "" for node in item.findall(".//m:t", NS))
            for item in shared_root.findall("m:si", NS)
        ]
        sheet_root = ET.fromstring(archive.read("xl/worksheets/sheet1.xml"))
    rows = []
    for row in sheet_root.findall(".//m:row", NS):
        values = {}
        for cell in row.findall("m:c", NS):
            node = cell.find("m:v", NS)
            if node is None:
                continue
            values[column_index(cell.get("r"))] = shared[int(node.text)] if cell.get("t") == "s" else node.text
        rows.append([values.get(index, "") for index in range(max(values, default=-1) + 1)])
    header = rows[13]
    return [
        {header[index]: row[index] if index < len(row) else "" for index in range(len(header))}
        for row in rows[14:]
    ]

assert DB.exists()
assert all((KEDI_DIR / f"{year}_kedi_higher_education_school.xlsx").exists() for year in (2024, 2025))
con = duckdb.connect(str(DB), read_only=True)
print("Sources loaded for", OPEN_ID)

Sources loaded for 1042688965


## Results

### 2. Confirm EDSS coverage and 2025 residual values

In [2]:
tables = con.execute('''
SELECT table_schema, table_name
FROM information_schema.columns
WHERE column_name = '개방ID'
  AND table_schema IN ('higher_education', 'university_disclosure')
GROUP BY ALL
ORDER BY ALL
''').fetchall()

panel_summary = []
nonzero_values = []
for schema_name, table_name in tables:
    cursor = con.execute(
        f"SELECT * FROM {schema_name}.{table_name} WHERE 개방ID=? AND 조사년도='2025'",
        [OPEN_ID],
    )
    rows = cursor.fetchall()
    if not rows:
        continue
    columns = [item[0] for item in cursor.description]
    panel_summary.append((f"{schema_name}.{table_name}", len(rows)))
    for index, column in enumerate(columns):
        if column.startswith("_") or column in DIMENSIONS:
            continue
        for value in sorted({str(row[index]) for row in rows if numeric_nonzero(row[index])}):
            nonzero_values.append((f"{schema_name}.{table_name}", column, value))

panel_count = len(panel_summary)
row_count = sum(row_count for _, row_count in panel_summary)
display(Markdown(f"**2025 coverage:** {panel_count} panels, {row_count} rows"))
display(panel_summary)
display(Markdown("**All non-zero measure/value combinations in 2025:**"))
display(nonzero_values)

**2025 coverage:** 10 panels, 76 rows

[('higher_education.panel_0101', 1),
 ('higher_education.panel_0104', 1),
 ('higher_education.panel_0105', 1),
 ('higher_education.panel_0231', 4),
 ('higher_education.panel_0236', 1),
 ('higher_education.panel_0246', 12),
 ('university_disclosure.panel_0306', 3),
 ('university_disclosure.panel_0308', 5),
 ('university_disclosure.panel_0715', 36),
 ('university_disclosure.panel_1017', 12)]

**All non-zero measure/value combinations in 2025:**

[('higher_education.panel_0101', '고등교육학교_학교수', '1'),
 ('higher_education.panel_0101', '야간학교수', '1'),
 ('higher_education.panel_0101', '공학학교수', '1'),
 ('higher_education.panel_0104', '대학원개황_학교수', '1'),
 ('higher_education.panel_0236', '본교학교수', '1'),
 ('higher_education.panel_0236', '학교수', '1'),
 ('university_disclosure.panel_0715', '등록금_대학원_입학정원수', '6'),
 ('university_disclosure.panel_0715', '등록금_대학원_수업료', '3967000.0')]

### 3. Verify school-level activity is zero and inspect department residue

In [3]:
school_columns = [
    "조사년도", "지역명", "본분교명", "고등교육학교_입학생수", "고등교육학교_여자입학생수",
    "고등교육학교_졸업생수", "고등교육학교_여자졸업생수", "고등교육학교_교원수",
    "고등교육학교_여자교원수", "고등교육학교_사무직원수", "고등교육학교_여자사무직원수",
    "고등교육학교_재적학생수", "고등교육학교_재적여학생수", "고등교육학교_학과수",
]
school_rows = [dict(zip(school_columns, row)) for row in con.execute(
    f"SELECT {','.join(school_columns)} FROM higher_education.panel_0101 WHERE 개방ID=? ORDER BY 조사년도",
    [OPEN_ID],
).fetchall()]

department_rows = [dict(zip(
    ["조사년도", "단과대학명", "학과명", "학과상태명", "학위과정구분명", "주야간계절구분명"], row
)) for row in con.execute('''
SELECT DISTINCT 조사년도, 단과대학명, 학과명, 학과상태명, 학위과정구분명, 주야간계절구분명
FROM university_disclosure.panel_1017
WHERE 개방ID=?
ORDER BY 조사년도, 학과명, 학위과정구분명
''', [OPEN_ID]).fetchall()]

display(school_rows)
display(department_rows)

[{'조사년도': '2024',
  '지역명': '서울 성북구',
  '본분교명': '본교',
  '고등교육학교_입학생수': '0',
  '고등교육학교_여자입학생수': '0',
  '고등교육학교_졸업생수': '0',
  '고등교육학교_여자졸업생수': '0',
  '고등교육학교_교원수': '0',
  '고등교육학교_여자교원수': '0',
  '고등교육학교_사무직원수': '0',
  '고등교육학교_여자사무직원수': '0',
  '고등교육학교_재적학생수': '0',
  '고등교육학교_재적여학생수': '0',
  '고등교육학교_학과수': '0'},
 {'조사년도': '2025',
  '지역명': '서울 성북구',
  '본분교명': '본교(제1캠퍼스)',
  '고등교육학교_입학생수': '0',
  '고등교육학교_여자입학생수': '0',
  '고등교육학교_졸업생수': '0',
  '고등교육학교_여자졸업생수': '0',
  '고등교육학교_교원수': '0',
  '고등교육학교_여자교원수': '0',
  '고등교육학교_사무직원수': '0',
  '고등교육학교_여자사무직원수': '0',
  '고등교육학교_재적학생수': '0',
  '고등교육학교_재적여학생수': '0',
  '고등교육학교_학과수': '0'}]

[{'조사년도': '2025',
  '단과대학명': '문화관광대학',
  '학과명': '창업학과',
  '학과상태명': '기존',
  '학위과정구분명': '박사',
  '주야간계절구분명': '야간'},
 {'조사년도': '2025',
  '단과대학명': '문화관광대학',
  '학과명': '창업학과',
  '학과상태명': '기존',
  '학위과정구분명': '석박사통합',
  '주야간계절구분명': '야간'},
 {'조사년도': '2025',
  '단과대학명': '문화관광대학',
  '학과명': '창업학과',
  '학과상태명': '기존',
  '학위과정구분명': '석사',
  '주야간계절구분명': '야간'},
 {'조사년도': '2025',
  '단과대학명': '문화관광대학',
  '학과명': '회계세무학과',
  '학과상태명': '기존',
  '학위과정구분명': '박사',
  '주야간계절구분명': '야간'},
 {'조사년도': '2025',
  '단과대학명': '문화관광대학',
  '학과명': '회계세무학과',
  '학과상태명': '기존',
  '학위과정구분명': '석박사통합',
  '주야간계절구분명': '야간'},
 {'조사년도': '2025',
  '단과대학명': '문화관광대학',
  '학과명': '회계세무학과',
  '학과상태명': '기존',
  '학위과정구분명': '석사',
  '주야간계절구분명': '야간'}]

### 4. Cross-check the exact KEDI candidate

In [4]:
candidate_fields = [
    "학교코드", "학교명", "학제", "대학원구분", "학교상태", "설립", "시군구", "본분교",
    "재적생_전체_계", "입학자_전체_계", "졸업자_전체_계", "전임교원_계", "직원_계", "학과수_전체",
]
kedi_candidates = {}
for year in (2024, 2025):
    candidates = [
        {key: row.get(key, "") for key in candidate_fields}
        for row in xlsx_records(KEDI_DIR / f"{year}_kedi_higher_education_school.xlsx")
        if row.get("시군구", "").startswith("서울 성북구")
        and row.get("대학원구분") == "부설대학원"
        and "AI세무" in row.get("학교명", "")
    ]
    kedi_candidates[year] = candidates

display(kedi_candidates)

{2024: [{'학교코드': '',
   '학교명': '성신여자대학교 AI세무?회계대학원',
   '학제': '특수대학원',
   '대학원구분': '부설대학원',
   '학교상태': '기존',
   '설립': '사립',
   '시군구': '서울 성북구',
   '본분교': '본교(제1캠퍼스)',
   '재적생_전체_계': '0',
   '입학자_전체_계': '0',
   '졸업자_전체_계': '3',
   '전임교원_계': '0',
   '직원_계': '0',
   '학과수_전체': '0'}],
 2025: [{'학교코드': '53068G42',
   '학교명': '성신여자대학교 AI세무?회계대학원',
   '학제': '특수대학원',
   '대학원구분': '부설대학원',
   '학교상태': '기존',
   '설립': '사립',
   '시군구': '서울 성북구',
   '본분교': '본교(제1캠퍼스)',
   '재적생_전체_계': '0',
   '입학자_전체_계': '0',
   '졸업자_전체_계': '0',
   '전임교원_계': '0',
   '직원_계': '0',
   '학과수_전체': '0'}]}

### 5. Run decision-critical checks

In [5]:
assert panel_count == 10
assert row_count == 76
assert {row["조사년도"] for row in school_rows} == {"2024", "2025"}

activity_columns = school_columns[3:]
assert all(str(row[column]) == "0" for row in school_rows for column in activity_columns)
assert {row["학과명"] for row in department_rows} == {"창업학과", "회계세무학과"}
assert {row["학과상태명"] for row in department_rows} == {"기존"}
assert {row["주야간계절구분명"] for row in department_rows} == {"야간"}

allowed_nonzero = {
    "고등교육학교_학교수", "야간학교수", "공학학교수", "대학원개황_학교수",
    "본교학교수", "학교수", "등록금_대학원_입학정원수", "등록금_대학원_수업료",
}
assert {column for _, column, _ in nonzero_values} == allowed_nonzero
assert len(nonzero_values) == 8

assert len(kedi_candidates[2024]) == 1
assert len(kedi_candidates[2025]) == 1
assert kedi_candidates[2025][0]["학교코드"] == "53068G42"
assert kedi_candidates[2025][0]["학교상태"] == "기존"
assert kedi_candidates[2024][0]["졸업자_전체_계"] == "3"
assert all(
    kedi_candidates[2025][0][column] == "0"
    for column in ["재적생_전체_계", "입학자_전체_계", "졸업자_전체_계", "전임교원_계", "직원_계", "학과수_전체"]
)
print("All decision-critical checks passed.")

All decision-critical checks passed.


### 6. Official closure evidence

- [성신여자대학교 공식 연혁](https://www.sungshin.ac.kr/main_kor/10918/subview.do): `2024년 09월 01일 | AI세무·회계대학원 폐지`
- [성신여자대학교 학칙 개정 이력](https://rule.sungshin.ac.kr/service/law/joHistoryContent.do?contentSeq=83688&endDate=20251023&seq=17&startDate=20090301): 2024-09-01 시행 조문에서 AI세무·회계대학원이 특수대학원 목록에서 삭제됨.

KEDI 2025의 `학교상태=기존`과 EDSS 학과상태 `기존`은 이 공식 폐지일보다 뒤의 상태이므로, 활성 여부를 판단할 때는 공식 이력을 우선한다.

## Takeaways

1. **신원:** `1042688965`는 성신여자대학교 AI세무·회계대학원과 일치한다. 2025 KEDI 학교코드는 `53068G42`다.
2. **실적:** 2025 EDSS 10개 패널 76행에 학생·입학·졸업·교원·직원·학과 수는 모두 0이다.
3. **잔존 정보:** 입학정원 6명, 수업료 3,967,000원, 창업학과·회계세무학과 `기존` 표시는 남아 있다.
4. **공식 상태:** 대학원은 2024-09-01 폐지되었다. KEDI/EDSS의 `기존` 표시는 후행 정리 지연으로 보는 것이 가장 일관적이다.
5. **권고:** `confirmed_identity_closed_school_residual_id`로 분류하고 활성기관 분석에서 제외한다. 수동 연결은 `53068G42`로 하되 공식 폐지 상태로 덮어쓴다.